In [26]:
import torch 
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import BertModel, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
import random

In [25]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [24]:
def compute_metrics(y_true,y_pred):
    acc = accuracy_score(y_true,y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    return f1, acc

# Custom Dataset and Dataloader

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, tokenizer, max_length, sample=2):
        self.data = load_dataset("tblard/allocine")
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.data["train"])
    def __getitem__(self, idx):
        training_data = self.data["train"]
        item = training_data[idx]
        review = item["review"]
        labels = item["label"]
        encoding = self.tokenizer(
            review,
            truncation=True,
            padding="max_length",
            max_length = self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids":encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

In [17]:
def create_dataloaders(
    dataset,
    batch_size=16,
    train_ratio=0.8,
    seed=42,
    sampler=None):
    train_size = int(train_ratio*len(dataset))
    val_size = len(dataset) - train_size

    generator = torch.Generator().manual_seed(seed)

    train_dataset, val_dataset = random_split(dataset,
        [train_size, val_size],
        generator = generator)
    
    train_sampler = None
    if sampler is not None:
        train_weigths = sampler[train_dataset.indices]
        train_sampler = WeightedRandomSampler(
            train_weigths, len(train_dataset))
    train_loader  = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle= (train_sampler is None),
        sampler = train_sampler
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )
    return train_loader, val_loader

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
allocine_dataset = CustomDataset(bert_tokenizer, max_length=256)

In [ ]:
train_loader, val_loader = create_dataloaders(allocine_dataset)

In [ ]:
print(f"Number of batches in train_loader: {len(train_loader)}")

for i, batch in enumerate(train_loader):
    if i < 2:  # 2 batches
        print(f"\n--- Batch {i+1} ---")
        print("Input IDs shape:", batch["input_ids"].shape)
        print("Attention Mask shape:", batch["attention_mask"].shape)
        print("Labels shape:", batch["labels"].shape)
        print("Example Input IDs:", batch["input_ids"][0][:10]) # First 10 tokens of the first sample
        print("Example Labels:", batch["labels"][0])
    else:
        break

# Model creation

In [7]:
class BertClassifier(nn.Module):
    def __init__(self,model_name ,n_classes):
        super().__init__()
        self.pretrained = BertModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.pretrained.config.hidden_size, n_classes)
    def forward(self, input_ids, attention_mask):
        outputs = self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)
        return logits
    

In [22]:
def train_epochs(model, optimizer, criterion, train_loader, device="cpu"):
  train_loss = 0
  all_preds = []
  all_labels = []
  total_sample = 0
  model.train()

  progress_bar = tqdm.tqdm(train_loader, desc="Training",leave=False)
  for batch in progress_bar:
    optimizer.zero_grad()
    labels = batch["labels"].to(device)
    inputs = batch['input_ids'].to(device)
    attention_mask = batch["attention_mask"].to(device)
    output = model(inputs, attention_mask)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    b = labels.size(0)
    preds = output.argmax(dim=1)
    total_sample += b
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(preds.cpu().numpy())
    progress_bar.set_postfix(loss=f"{train_loss/total_sample:.4f}")

  if total_sample == 0:
    return 0.0, 0.0, 0.0
  avg_loss = train_loss/total_sample
  train_acc,_ = compute_metrics(all_labels, all_preds)
  return avg_loss, train_acc



In [23]:
def eval_epoch(model, criterion, eval_loader, device="cpu"):
    val_loss = 0.0
    correct = 0.0
    total_sample = 0.0
    all_preds = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        progress_bar = tqdm(eval_loader, desc="Validation", unit="batch")
        for batch in progress_bar:
            inputs = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            output = model(inputs, attention_mask=attention_mask)
            loss = criterion(output, labels)
            b = labels.size(0)
            val_loss += loss.item() * b
            preds = output.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy()) 
            all_labels.extend(labels.cpu().numpy())
            total_sample += b
            progress_bar.set_postfix(loss=f"{val_loss/total_sample:.4f}")
        if total_sample == 0:
            return 0.0, 0.0, 0.0
        avg_loss = val_loss / total_sample
        acc, f1 = compute_metrics(all_preds, all_labels)
        return avg_loss, acc, f1


In [ ]:
model_name = "bert-base-uncased"
model = BertClassifier(model_name, n_classes=3)
tokenizer = AutoTokenizer.from_pretrained(model_name)